In [ ]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
import pandas as pd
from fedgnn.analysis.results import process_results_folder_json
from fedgnn.analysis.training_logs import parse_client_csv, process_fp_logs


In [ ]:
import pandas as pd
from fedgnn.analysis.results import process_results_folder_json
from fedgnn.analysis.training_logs import parse_client_csv, process_fp_logs

In [ ]:
import os
import pathlib

def find_repo_root(start=None):
    p = pathlib.Path(start or os.getcwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / "conf" / "base.yaml").exists():
            return parent
    return p

PROJECT_ROOT = find_repo_root()

no_pe_results_path  = str(PROJECT_ROOT / "results" / "test_ablation_6")

results_df = process_results_folder_json(no_pe_results_path)

print(results_df.head())


## Result Analysis

### Considering No PE

In [ ]:
no_pe_results_path  = str(PROJECT_ROOT / "results" / "test_ablation_6")

In [ ]:
no_pe_results_df = process_results_folder_json(no_pe_results_path)
no_pe_results_df

In [ ]:
pe_results_path = str(PROJECT_ROOT / "src" / "results" / "multi_ablation_test_100_rounds_PE")

In [ ]:
pe_results_df = process_results_folder_json(pe_results_path)
pe_results_df



In [ ]:
# if any row has nan, drop it for pe_results_df
pe_results_df = pe_results_df[~pe_results_df.isna().any(axis=1)]
pe_results_df
# reset the index
pe_results_df = pe_results_df.reset_index(drop=True)
pe_results_df






#### lets drop unused columns 

In [ ]:
# lets drop the following columsn aon both: device, model_type, num clients, hop, global_results, cliet_results,
# and then drop the rows where test_acc_mean is 0
# no_pe_results_df = no_pe_results_df.drop(columns=['device', 'model_type', 'num_clients', 'hop', 'global_results', 'client_results','experiment_id' 'fulltraining_flag' ])
no_pe_results_df = no_pe_results_df.drop('fulltraining_flag', axis=1)
no_pe_results_df






In [ ]:
pe_results_df = pe_results_df.drop(columns=['device', 'model_type', 'num_clients', 'hop', 'global_results', 'client_results','experiment_id', 'fulltraining_flag' ])


pe_results_df



In [ ]:
# lets add a columna called pe where all values are True for pe_results_df
pe_results_df['pe'] = True
pe_results_df





In [ ]:
# lets add a columna called pe where all values are False for no_pe_results_df
no_pe_results_df['pe'] = False
no_pe_results_df





In [ ]:
df = pd.concat([no_pe_results_df, pe_results_df], ignore_index=True)
df
# drop the columns: average_global_result, std_global
df = df.drop(columns=['average_global_result', 'std_global'])
df


In [ ]:
# get dfs for each dataset based on the dataset column & reset the index
df_cora = df[df['dataset'] == 'Cora']
df_cora = df_cora.reset_index(drop=True)
df_cora


In [ ]:
df_pubmed = df[df['dataset'] == 'Pubmed']
df_pubmed = df_pubmed.reset_index(drop=True)
df_pubmed

In [ ]:
df_citeseer = df[df['dataset'] == 'Citeseer']
df_citeseer = df_citeseer.reset_index(drop=True)
df_citeseer

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def plot_client_accuracy_by_pe_and_beta(df):
    """
    Creates a grouped bar plot showing average client accuracy 
    for each data loading method, split by PE status and faceted by beta.

    Parameters:
    - df (pd.DataFrame): A DataFrame with columns:
        ['data_loading_option', 'average_client_result', 'pe', 'beta']

    Returns:
    - Seaborn FacetGrid plot
    """
    g = sns.catplot(
        data=df,
        x="data_loading_option",
        y="average_client_result",
        hue="pe",
        col="beta",
        kind="bar",
        height=5,
        aspect=1,
        palette="Set2"
    )
    g.set_axis_labels("Data Loading Option", "Average Client Accuracy")
    g.set_titles("Beta = {col_name}")
    g._legend.set_title("Positional Encoding")

    for ax in g.axes.flat:
        ax.set_ylim(0.6, 0.85)
        ax.tick_params(axis='x', rotation=45)

    plt.tight_layout()
    return g


In [ ]:
plot_client_accuracy_by_pe_and_beta(df_cora)

In [ ]:
plot_client_accuracy_by_pe_and_beta(df_pubmed)

In [ ]:
plot_client_accuracy_by_pe_and_beta(df_citeseer)

In [ ]:
import pandas as pd

def make_accuracy_std_table(df):
    """
    Returns a pivot table with average client accuracy and standard deviation (±) 
    per data loading method, grouped by PE status and beta value.

    Parameters:
    - df (pd.DataFrame): Must include columns:
        ['data_loading_option', 'beta', 'pe', 'average_client_result', 'std_client']

    Returns:
    - pd.DataFrame: Pivot table with formatted accuracy ± std
    """

    # Create a row label for grouping
    df["row_label"] = df.apply(lambda row: f"{'PE' if row.pe else 'NoPE'}_beta{row.beta}", axis=1)

    # Format values with ±
    df["accuracy_std_fmt"] = df.apply(
        lambda row: f"{row.average_client_result:.4f} ± {row.std_client:.4f}", axis=1
    )

    # Pivot the table
    table = df.pivot(index="row_label", columns="data_loading_option", values="accuracy_std_fmt")

    return table


In [ ]:
formatted_table = make_accuracy_std_table(df_cora)
formatted_table

In [ ]:
formatted_table_citeseer = make_accuracy_std_table(df_citeseer)
formatted_table_citeseer 

In [ ]:
formatted_table_pubmed= make_accuracy_std_table(df_pubmed)
formatted_table_pubmed 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def plot_avg_client_results_subsets(df):
    """
    Generates grouped bar plots of average client results by data loading option for
    each of the 4 combinations of (PE, beta).
    
    Parameters:
    - df (pd.DataFrame): Should include columns: ['dataset', 'data_loading_option', 'average_client_result', 'pe', 'beta']
    """

    # Define the four combinations to iterate over
    combinations = [
        {"pe": True, "beta": 1, "label": "IID + PE"},
        {"pe": False, "beta": 1, "label": "IID + NoPE"},
        {"pe": True, "beta": 10000, "label": "Non-IID + PE"},
        {"pe": False, "beta": 10000, "label": "Non-IID + NoPE"},
    ]

    for combo in combinations:
        sub_df = df[(df["pe"] == combo["pe"]) & (df["beta"] == combo["beta"])]
        sub_df = sub_df[sub_df["data_loading_option"] != "diffusion"]

        # Group and pivot
        grouped = sub_df.groupby(['dataset', 'data_loading_option'])['average_client_result'].mean().unstack()

        # Plotting
        ax = grouped.plot(kind='bar', figsize=(10, 6))
        ax.set_ylabel("Average Client Result")
        ax.set_title(f"Average Client Results by Loading Option ({combo['label']})")
        ax.legend(title="Data Loading Option")
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.show()


In [ ]:
plot_avg_client_results_subsets(df)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def plot_and_summarize_without_diffusion(df):
    """
    For each combination of PE and beta, plot average client results
    by data loading option (excluding 'diffusion') and print summary tables.
    
    Parameters:
    - df (pd.DataFrame): Must contain columns:
        ['dataset', 'data_loading_option', 'average_client_result', 'pe', 'beta']
    """

    combinations = [
        {"pe": True, "beta": 1, "label": "IID + PE"},
        {"pe": False, "beta": 1, "label": "IID + NoPE"},
        {"pe": True, "beta": 10000, "label": "Non-IID + PE"},
        {"pe": False, "beta": 10000, "label": "Non-IID + NoPE"},
    ]

    for combo in combinations:
        # Subset the data
        sub_df = df[(df["pe"] == combo["pe"]) & (df["beta"] == combo["beta"])]
        sub_df = sub_df[sub_df["data_loading_option"] != "diffusion"]

        # Group and pivot for plotting
        grouped = sub_df.groupby(['dataset', 'data_loading_option'])['average_client_result'].mean().unstack()

        # Plotting
        ax = grouped.plot(kind='bar', figsize=(10, 6))
        ax.set_ylabel("Average Client Result")
        ax.set_title(f"Average Client Results (No 'diffusion') — {combo['label']}")
        ax.legend(title="Data Loading Option")
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.show()

        # Print summary table
        print(f"\nSummary Table — {combo['label']}")
        print(grouped.round(4))
        print("-" * 60)


In [ ]:
plot_and_summarize_without_diffusion(df)

In [ ]:
def make_accuracy_std_tables_by_condition(df):
    """
    Generates four tables (one for each of the four PE/beta combinations)
    showing average client accuracy ± std, grouped by dataset and data_loading_option.

    Parameters:
    - df (pd.DataFrame): Must include columns:
        ['dataset', 'data_loading_option', 'average_client_result', 'std_client', 'pe', 'beta']

    Returns:
    - dict of DataFrames: keys are condition labels (e.g., 'PE_beta1'), values are formatted tables
    """
    tables = {}
    combinations = [
        {"pe": True, "beta": 1, "label": "PE_beta1"},
        {"pe": False, "beta": 1, "label": "NoPE_beta1"},
        {"pe": True, "beta": 10000, "label": "PE_beta10000"},
        {"pe": False, "beta": 10000, "label": "NoPE_beta10000"},
    ]

    for combo in combinations:
        sub_df = df[(df["pe"] == combo["pe"]) & (df["beta"] == combo["beta"])].copy()
        sub_df["accuracy_std_fmt"] = sub_df.apply(
            lambda row: f"{row.average_client_result:.4f} ± {row.std_client:.4f}", axis=1
        )
        table = sub_df.pivot(index="dataset", columns="data_loading_option", values="accuracy_std_fmt")
        tables[combo["label"]] = table

    return tables


In [ ]:
tables = make_accuracy_std_tables_by_condition(df)

In [ ]:
# Example to print one:
print("=== IID + PE ===")
(tables["PE_beta1"])

In [ ]:
def make_long_table_by_dataset(df):
    """
    Returns a long-format table with rows as (Distribution, Encoding, Propagation Technique),
    and columns as datasets. Each cell is 'mean ± std' for client accuracy.

    Parameters:
    - df (pd.DataFrame): Must include columns:
        ['dataset', 'data_loading_option', 'average_client_result', 'std_client', 'pe', 'beta']

    Returns:
    - pd.DataFrame: Pivot table with formatted values and datasets as columns
    """
    # Copy and prepare readable labels
    df = df.copy()
    df["Distribution"] = df["beta"].map({1: "IID", 10000: "Non-IID"})
    df["Encoding"] = df["pe"].map({True: "PE", False: "NoPE"})
    df["Propagation Technique"] = df["data_loading_option"].str.capitalize()

    # Format mean ± std string
    df["Formatted"] = df.apply(
        lambda row: f"{row.average_client_result:.4f} ± {row.std_client:.4f}", axis=1
    )

    # Create pivot table
    table = df.pivot_table(
        index=["Distribution", "Encoding", "Propagation Technique"],
        columns="dataset",
        values="Formatted",
        aggfunc="first"  # each group should be unique
    )

    # Sort for clarity
    table = table.sort_index()

    return table


In [ ]:
long_table = make_long_table_by_dataset(df)
long_table

## Measuring the training results

In [ ]:
data = parse_client_csv(str(PROJECT_ROOT / "src" / "results" / "multi_ablation_test_100_rounds_NO_PE" / "Citeseer_adjacency_GCN_beta1_clients10" / "training_Citeseer_adjacency_GCN_beta1_clients10_20250504_183037.csv"))

In [ ]:
data.keys()

In [ ]:
import matplotlib.pyplot as plt

def plot_all_metrics(data_dict):
    """
    Loops through all dataframes in the parse_client_csv output and plots them.

    Parameters:
    - data_dict (dict): Output from parse_client_csv
    """
    for key, df in data_dict.items():
        if isinstance(df, pd.DataFrame):
            plt.figure(figsize=(12, 5))
            df.plot(ax=plt.gca())
            plt.title(f"{key}")
            plt.xlabel("Step" if "step" in key else "Epoch")
            plt.ylabel("Loss/Accuracy")
            plt.legend(loc='best')
            plt.grid(True)
            plt.tight_layout()
            plt.show()


In [ ]:
plot_all_metrics(data)

In [ ]:
data['loss_df_step']
# add an average column
data['loss_df_step']['average'] = data['loss_df_step'].mean(axis=1)
data['loss_df_step']
# plot the loss_df_step
data['loss_df_step']['average'].plot()

In [ ]:
# plot the loss_df_step
data['loss_df_step'].plot()
plt.title('Loss of Non-IID Data')
plt.xlabel('Step')
plt.ylabel('Loss')
# increate size of the figure
plt.figure(figsize=(10, 6))
plt.show()



In [ ]:
data_iid_path = str(PROJECT_ROOT / "src" / "results" / "multi_ablation_test_100_rounds_NO_PE" / "Citeseer_adjacency_GCN_beta10000_clients10" / "training_Citeseer_adjacency_GCN_beta10000_clients10_20250504_184001.csv")

In [ ]:
data_iid = parse_client_csv(data_iid_path)
data_iid.keys()
data_iid['loss_df_step']
# plot the loss_df_step
data_iid['loss_df_step'].plot()
# add title and labels
plt.title('Loss of IID Data')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.figure(figsize=(20, 10))
plt.show()

In [ ]:
# add an average column
data_iid['loss_df_step']['average'] = data_iid['loss_df_step'].mean(axis=1)
data_iid['loss_df_step']
# plot the average column
data_iid['loss_df_step']['average'].plot()
plt.title('Loss of IID Data')
plt.xlabel('Step')

In [ ]:
# plot both data iid and data non iid
data['loss_df_step']['average'][:10].plot()
data_iid['loss_df_step']['average'][:10].plot()
plt.title('Loss of IID and Non-IID Data')
# add legend
plt.legend(['Non-IID', 'IID'])
plt.xlabel('Step')
plt.ylabel('Loss')
plt.grid(True)
plt.figure(figsize=(20, 10))

In [ ]:
data_non_iid_pe_path = str(PROJECT_ROOT / "src" / "results" / "multi_ablation_test_100_rounds_PE" / "Citeseer_adjacency_GCN_beta1_clients10" / "training_Citeseer_adjacency_GCN_beta1_clients10_20250504_144515.csv")

In [ ]:
data_iid_pe_path = str(PROJECT_ROOT / "src" / "results" / "multi_ablation_test_100_rounds_PE" / "Citeseer_adjacency_GCN_beta10000_clients10" / "training_Citeseer_adjacency_GCN_beta10000_clients10_20250504_145219.csv")

In [ ]:
data_non_iid_pe = parse_client_csv(data_non_iid_pe_path)
data_iid_pe = parse_client_csv(data_iid_pe_path)
data_non_iid_pe.keys()
data_iid_pe.keys()
data_non_iid_pe['loss_df_step']
data_iid_pe['loss_df_step']

In [ ]:
import matplotlib.pyplot as plt

def plot_loss_curves(*datasets, labels, title="Client Loss Comparison", figsize=(12, 6)):
    """
    Plots the average loss over training steps for multiple datasets.

    Parameters:
    - datasets: unpacked list of dicts (each from parse_client_csv)
    - labels: list of string labels corresponding to each dataset
    - title: title of the plot
    - figsize: size of the plot
    """
    plt.figure(figsize=figsize)

    for data, label in zip(datasets, labels):
        if 'average' not in data['loss_df_step'].columns:
            data['loss_df_step']['average'] = data['loss_df_step'].mean(axis=1)
        plt.plot(data['loss_df_step']['average'], label=label)

    plt.title(title, fontsize=14)
    plt.xlabel("Step", fontsize=12)
    plt.ylabel("Loss", fontsize=12)
    plt.legend(fontsize=11)
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_loss_curves(
    data, data_iid, data_non_iid_pe, data_iid_pe,
    labels=["Non-IID", "IID", "Non-IID + PE", "IID + PE"],
    title="Average Client Loss Over Steps"
)


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import linregress

def evaluate_loss_convergence(loss_series: pd.Series):
    """
    Evaluates monotonicity, average convergence rate, and log-scale convergence of a loss series.

    Parameters:
    - loss_series (pd.Series): A pandas Series of loss values over training steps

    Returns:
    - dict with:
        - 'monotonicity_score': Fraction of steps where loss does not increase
        - 'avg_convergence_rate': Average % decrease per step
        - 'log_slope': Slope of log(loss) over time
        - 'log_r2': R-squared of log-linear fit (fit quality)
    """
    loss = loss_series.dropna().values
    if len(loss) < 2:
        return {
            'monotonicity_score': None,
            'avg_convergence_rate': None,
            'log_slope': None,
            'log_r2': None
        }

    # 1. Monotonicity
    diffs = loss[:-1] - loss[1:]
    monotonicity = np.mean(diffs >= 0)

    # 2. Step-wise convergence rate
    pct_decreases = np.where(loss[:-1] != 0, diffs / loss[:-1], 0)
    convergence_rate = np.mean(pct_decreases)

    # 3. Log-scale slope and R²
    positive_loss = loss[loss > 0]
    steps = np.arange(len(positive_loss))
    log_loss = np.log(positive_loss)

    if len(log_loss) < 2:
        log_slope = None
        log_r2 = None
    else:
        slope, intercept, r_value, _, _ = linregress(steps, log_loss)
        log_slope = slope
        log_r2 = r_value ** 2

    return {
        'monotonicity_score': round(float(monotonicity), 4),
        'avg_convergence_rate': round(float(convergence_rate), 4),
        'log_slope': round(float(log_slope), 6) if log_slope is not None else None,
        'log_r2': round(float(log_r2), 4) if log_r2 is not None else None
    }


In [ ]:
data_points = [data, data_iid, data_non_iid_pe, data_iid_pe]
labels = ["Non-IID", "IID", "Non-IID + PE", "IID + PE"]

for data_point, label in zip(data_points, labels):
    results = evaluate_loss_convergence(data_point['loss_df_step']['average'])
    print(f"{label}:")
    print(results)
    print("-" * 40)



In [ ]:
# Re-import needed modules and re-define the function after execution state reset
import numpy as np
import pandas as pd

def evaluate_brachistochrone_efficiency(loss_series: pd.Series, epsilon: float = 0.1):
    """
    Evaluates two Brachistochrone-inspired metrics for convergence:
    - BES: Brachistochrone Efficiency Score (fit to ideal exponential decay)
    - NDT: Normalized Descent Time to reach epsilon * initial loss

    Parameters:
    - loss_series (pd.Series): Series of training loss values
    - epsilon (float): Threshold to define early descent (default = 10% of initial loss)

    Returns:
    - dict with:
        - 'BES': Brachistochrone Efficiency Score (0–1)
        - 'NDT': Normalized Descent Time (0–1)
    """
    loss = loss_series.dropna().values
    T = len(loss)
    if T < 2:
        return {'BES': None, 'NDT': None}

    L0 = loss[0]
    LT = loss[-1]

    # 1. Ideal exponential decay curve
    k = -np.log(LT / L0) / T if LT > 0 and L0 > 0 else 0
    t = np.arange(T)
    ideal = L0 * np.exp(-k * t)

    # 2. Brachistochrone Efficiency Score (area difference normalized by L0)
    deviation = np.abs(loss - ideal) / L0
    BES = 1 - np.mean(deviation)

    # 3. Normalized Descent Time to reach epsilon * L0
    threshold = L0 * epsilon
    below_thresh_indices = np.where(loss <= threshold)[0]
    if len(below_thresh_indices) > 0:
        t_eps = below_thresh_indices[0]
        NDT = t_eps / T
    else:
        NDT = 1.0  # never reached threshold

    return {
        'BES': round(float(BES), 4),
        'NDT': round(float(NDT), 4)
    }


In [ ]:
data_points = [data, data_iid, data_non_iid_pe, data_iid_pe]
labels = ["Non-IID", "IID", "Non-IID + PE", "IID + PE"]

for data_point, label in zip(data_points, labels):
    results = evaluate_brachistochrone_efficiency(data_point['loss_df_step']['average'])
    print(f"{label}:")
    print(results)
    print("-" * 40)

In [ ]:
results = evaluate_loss_convergence(data['loss_df_step']['average'])
print(results)

In [ ]:
def evaluate_smoothed_convergence(loss_series: pd.Series, window: int = 10):
    """
    Applies the original convergence rate logic on a smoothed version of the loss curve.

    Parameters:
    - loss_series (pd.Series): Raw training loss
    - window (int): Smoothing window size (in steps)

    Returns:
    - dict with:
        - 'monotonicity_score'
        - 'avg_convergence_rate'
    """
    smoothed = loss_series.rolling(window=window, min_periods=1).mean().dropna().values
    T = len(smoothed)
    if T < 2:
        return {'monotonicity_score': None, 'avg_convergence_rate': None}

    diffs = smoothed[:-1] - smoothed[1:]
    monotonicity = np.mean(diffs >= 0)
    pct_decreases = np.where(smoothed[:-1] != 0, diffs / smoothed[:-1], 0)
    convergence_rate = np.mean(pct_decreases)

    return {
        'monotonicity_score': round(float(monotonicity), 4),
        'avg_convergence_rate': round(float(convergence_rate), 4)*300
    }


In [ ]:
data_points = [data, data_iid, data_non_iid_pe, data_iid_pe]
labels = ["Non-IID", "IID", "Non-IID + PE", "IID + PE"]

for data_point, label in zip(data_points, labels):
    results = evaluate_smoothed_convergence(data_point['loss_df_step']['average'])
    print(f"{label}:")
    print(results)
    print("-" * 40)

In [ ]:
# plots
data_non_iid_pe['loss_df_step'].plot()
plt.title('Loss of Non-IID Data with PE')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.figure(figsize=(20, 10))
plt.show()


In [ ]:
data_iid_pe['loss_df_step'].plot()
plt.title('Loss of IID Data with PE')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.figure(figsize=(20, 10))
plt.show()


In [ ]:
import os
import re

def collect_training_csvs_named(folder_path):
    """
    Scans a folder and returns training CSV paths with standardized variable names.

    Parameters:
    - folder_path (str): Top-level folder containing subfolders with training CSVs.

    Returns:
    - List[Tuple[str, str]]: (variable_name, full_csv_path), where variable_name is like 'Citeseer_adjacency_beta1_pe'
    """
    pe_status = "nope" if "NO_PE" in folder_path.upper() else "pe"
    csv_files = []

    for subdir in os.listdir(folder_path):
        subdir_path = os.path.join(folder_path, subdir)

        if os.path.isdir(subdir_path):
            for file in os.listdir(subdir_path):
                if file.startswith("training_") and file.endswith(".csv"):
                    full_path = os.path.join(subdir_path, file)

                    # Extract parts using regex
                    match = re.search(r'training_(\w+?)_(\w+?)_GCN_beta(\d+)_clients\d+', file)
                    if match:
                        dataset, method, beta = match.groups()
                        var_name = f"{dataset}_{method}_beta{beta}_{pe_status}"
                        csv_files.append((var_name, full_path))

    return csv_files


In [ ]:
no_pe_path = str(PROJECT_ROOT / "src" / "results" / "multi_ablation_test_100_rounds_NO_PE")

files = collect_training_csvs_named(no_pe_path)

pe_path = str(PROJECT_ROOT / "src" / "results" / "multi_ablation_test_100_rounds_PE")
pe_files = collect_training_csvs_named(pe_path)


In [ ]:
files

In [ ]:
file_dict = dict(files)

In [ ]:
file_dict.keys()

In [ ]:
# pubmed_files = # pick a subset of the dicy where the key contains Pubmed
pubmed_files = [file for file in files if 'Pubmed' in file[0]]
pubmed_files


In [ ]:
p = file_dict['Pubmed_diffusion_beta1_nope']
parsed = parse_client_csv(p)
l_df = parsed["final_loss_df"]
l_df["average"] = l_df.mean(axis=1)
l_df["average"].plot()
plt.title('Loss of Pubmed_diffusion_beta1_nope')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.figure(figsize=(20, 10))
plt.show()

In [ ]:
import matplotlib.pyplot as plt

def plot_final_loss_average(files, label_filter="Pubmed"):
    """
    Plots average final loss per run for all files matching label_filter in name.

    Parameters:
    - files: List of (name, path) tuples
    - label_filter: string to match (e.g., 'Pubmed')
    """
    for name, path in files:
        if label_filter in name:
            parsed = parse_client_csv(path)
            l_df = parsed["final_loss_df"]
            l_df["average"] = l_df.mean(axis=1)

            plt.figure(figsize=(10, 4))
            l_df["average"].plot()
            plt.title(f'Final Loss – {name}')
            plt.xlabel('Client')
            plt.ylabel('Loss')
            plt.grid(True)
            plt.tight_layout()
            plt.show()

# Usage
plot_final_loss_average(files, label_filter="Pubmed")


In [ ]:
import matplotlib.pyplot as plt

def plot_combined_final_loss(files, label_filter="Pubmed"):
    """
    Plots average final loss curves for all matching runs in a single plot with legends.

    Parameters:
    - files: List of (name, path) tuples
    - label_filter: string to filter runs (e.g., 'Pubmed')
    """
    plt.figure(figsize=(14, 6))

    for name, path in files:
        if label_filter in name:
            parsed = parse_client_csv(path)
            l_df = parsed["final_loss_df"]
            l_df["average"] = l_df.mean(axis=1)

            plt.plot(l_df["average"], label=name)

    plt.title(f'Final Loss (Averaged per Client) – {label_filter} Runs')
    plt.xlabel('Client')
    plt.ylabel('Loss')
    plt.legend(fontsize=8, loc='upper right')
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_combined_final_loss(files, label_filter="Pubmed")


In [ ]:
plot_combined_final_loss(files, label_filter="Cora")

In [ ]:
plot_combined_final_loss(files, label_filter="Citeseer")

In [ ]:
pe_files_dict = dict(pe_files)

In [ ]:
plot_combined_final_loss(pe_files, label_filter="Pubmed")

In [ ]:
plot_combined_final_loss(pe_files, label_filter="Citeseer")
plot_combined_final_loss(pe_files, label_filter="Cora")


In [ ]:
pubmed_300 = str(PROJECT_ROOT / "src" / "results" / "publmed_300_rounds_NO_PE" / "Pubmed_zero_hop_GCN_beta10000_clients10" / "training_Pubmed_zero_hop_GCN_beta10000_clients10_20250512_021802.csv")

In [ ]:
pubmed_stats = parse_client_csv(pubmed_300)

In [ ]:
pubmed_stats.keys()

In [ ]:
pubmed_stats["final_loss_df"]

In [ ]:
pubmed_stats["final_loss_df"].plot()
plt.title('Loss of Pubmed_zero_hop_GCN_beta10000_clients10')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.figure(figsize=(20, 10))
plt.show()
